In [10]:
from BU_hydrogel.analysis.lib.analysis_params import dataset_dir, figure_dir_analysis, data_list
from BU_hydrogel.analysis.lib.data_io import DataIO
from BU_hydrogel.analysis.lib.display_tools import generate_raster_plots_session, generate_heatmaps_session, firing_rate_per_protocol_master, firing_rate_per_param
from utils import load_obj, make_figure, save_fig, update_subplot_titles
import pandas as pd
from BU_hydrogel.analysis.lib.analysis_tools import detect_preferred_electrode, get_params_protocol, params_abbreviation
from BU_hydrogel.project_colors import ProjectColors
import numpy as np

# Load dataset + dump as pickle to speedup future data loading
data_io = DataIO(dataset_dir)
data_io.load_session('2026-05-12 rat LE 1355 A', load_pickle=True, load_waveforms=False)
data_io.dump_as_pickle()

clrs = ProjectColors()


loadname = dataset_dir / f'{data_io.session_id}_cells.csv'
cells_df = pd.read_csv(loadname, header=[0, 1], index_col=0)

laser_powers = [5000, 6000, 7000, 8000]
burst_durations = [10, 30]
laser_prr = 4300

dmd_intensities = [50, 150, 250]

ref_rec_id = data_io.recording_ids[1]
print(ref_rec_id)


Loading pickled data (not from h5 file)
rec_2_2026-05-12_rat_1355_A_stim_bu_stim_high


In [21]:
data_io.train_df.dmd_burst_duration.unique()

array([nan, 20.], dtype=float32)

In [ ]:
def plot_cell_responses(cid, savename):
    pref_ec = detect_preferred_electrode(data_io, cells_df)
    electrode = pref_ec[ref_rec_id]['bu_stim_high'].loc[cid].ec

    if pd.isna(electrode):
        return
    cluster_data = load_obj(dataset_dir / 'bootstrapped' / f'bootstrap_{cid}.pkl')

    fig_x_domains ={
            1: [[0.1, 0.35], [0.4, 0.65], [0.7, 0.95]],
     }

    fig_y_domains = {
            1: [[0.1, 0.9], [0.1, 0.9], [0.1, 0.9]],
    }

    fig = make_figure(
        width=1.5,
        height=1,
        x_domains=fig_x_domains,
        y_domains=fig_y_domains,
        xticks=[], yticks=[],
    )

    min_fr = None
    max_fr = None

    for bd_i, bd in enumerate(burst_durations):



        pos = dict(row=1, col=bd_i+1)

        fig.add_scatter(
            x=[0, 0, bd, bd],
            y=[-100, 1000, 1000, -100],
            mode='lines',
            fill='toself',
            line=dict(color='red', width=0.0001),
            fillcolor='rgba(255, 0, 0, 0.3)',
            showlegend=False,
            **pos,
        )

        for l_pwr in laser_powers:

            # Extract train ids
            df = data_io.train_df.query(
                f'electrode == {electrode} and '
                f'laser_burst_duration == {bd} and '
                f'laser_pulse_repetition_rate == {laser_prr} and '
                f'laser_power == {l_pwr}'
            )
            assert len(df) == 1
            train_id = df.iloc[0].train_id

            # Extract data to plot
            bins = cluster_data[train_id].get('bins')
            firing_rate = cluster_data[train_id].get('firing_rate')
            clr = clrs.min_max_map(val=l_pwr, min_val=2000, max_val=8100)

            if min_fr is None or np.min(firing_rate) < min_fr:
                min_fr = np.min(firing_rate)
            if max_fr is None or np.max(firing_rate) > max_fr:
                max_fr = np.max(firing_rate)

            fig.add_scatter(
                x=bins,
                y=firing_rate,
                mode='lines',
                name=f'pwr: {l_pwr:.0f}',
                showlegend=True if pos['col'] == 1 else False,
                legendgroup='laser_specs',
                line=dict(color=clr, width=1.5),
                **pos,
            )

    pos = dict(row=1, col=3)

    fig.add_scatter(
        x=[0, 0, 20, 20],
        y=[-100, 1000, 1000, -100],
        mode='lines',
        fill='toself',
        line=dict(color='red', width=0.0001),
        fillcolor='rgba(255, 0, 0, 0.3)',
        showlegend=False,
        **pos,
    )
    for di in dmd_intensities:
        train_id = data_io.train_df.query(
            f'dmd_intensity == {di}'
        )
        assert len(train_id) == 1
        train_id = train_id.iloc[0].train_id

        # Extract data to plot
        bins = cluster_data[train_id].get('bins')
        firing_rate = cluster_data[train_id].get('firing_rate')
        clr = clrs.dmd_intensity(intensity=di)

        if min_fr is None or np.min(firing_rate) < min_fr:
            min_fr = np.min(firing_rate)
        if max_fr is None or np.max(firing_rate) > max_fr:
            max_fr = np.max(firing_rate)

        fig.add_scatter(
            x=bins,
            y=firing_rate,
            mode='lines',
            name=f'dmd pwr: {di:.0f}',
            showlegend=True,
            legendgroup='dmd_intensity',
            line=dict(color=clr, width=1.5),
            **pos,
        )


    for i in range(3):
        fig.update_yaxes(
            range=[min_fr, max_fr],
            title_text='fr [Hz]' if i == 0 else '',
            tickvals = np.arange(0, 200, 25),
            row=1,
            col=i+1,
        )

        fig.update_xaxes(
            title_text=f'time [ms]',
            tickvals=np.arange(-100, 300, 100),
            row=1,
            col=i+1,
        )

    fig = update_subplot_titles(fig,
                                x_domains=fig_x_domains,
                                y_domains=fig_y_domains,
                                subplot_titles={(1,1): 'laser, bd: 10', (1,2): 'laser, bd: 30', (1,3): 'dmd'})

    save_fig(fig, savename, display=False)


for cluster_id in data_io.cluster_ids:
    sname = figure_dir_analysis / data_io.session_id / 'firing_rate_charts' / cluster_id
    plot_cell_responses(cluster_id, sname)


